In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../Telecom_Customers_Churn.csv")

# Basic information
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print (df.columns)
df.head()



Rows: 7043
Columns: 21
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
#Module 1: Data Profiling & Quality
#Q1. Create a complete Data Quality Report showing data type, 
#null count, null percentage, distinct count, duplicate count, and min/max values where applicable.


def create_data_quality_report(df):
    
    report = pd.DataFrame(index=df.columns)
    
    # Data type
    report["Data_Type"] = df.dtypes.astype(str)
    
    # Null count
    report["Null_Count"] = df.isnull().sum()
    
    # Null percentage
    report["Null_Percentage"] = (
        df.isnull().mean() * 100
    ).round(2)
    
    # Distinct count
    report["Distinct_Count"] = df.nunique(dropna=True)
    
    # Duplicate count
    # Number of duplicated values for each column
    report["Duplicate_Count"] = (
        df.shape[0] - df.nunique(dropna=True)
    )
    
    # Min and Max
    report["Min_Value"] = np.nan
    report["Max_Value"] = np.nan
    
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            report.loc[column, "Min_Value"] = df[column].min()
            report.loc[column, "Max_Value"] = df[column].max()
        
        elif pd.api.types.is_datetime64_any_dtype(df[column]):
            report.loc[column, "Min_Value"] = df[column].min()
            report.loc[column, "Max_Value"] = df[column].max()
    
    return report


quality_report = create_data_quality_report(df)

quality_report

,Data_Type,Null_Count,Null_Percentage,Distinct_Count,Duplicate_Count,Min_Value,Max_Value
customerID,object,0,0.0,7043,0,NaN,NaN
gender,object,0,0.0,2,7041,NaN,NaN
SeniorCitizen,int64,0,0.0,2,7041,0.00,1.00
Partner,object,0,0.0,2,7041,NaN,NaN
Dependents,object,0,0.0,2,7041,NaN,NaN
tenure,int64,0,0.0,73,6970,0.00,72.00
PhoneService,object,0,0.0,2,7041,NaN,NaN
MultipleLines,object,0,0.0,3,7040,NaN,NaN
InternetService,object,0,0.0,3,7040,NaN,NaN
OnlineSecurity,object,0,0.0,3,7040,NaN,NaN


In [9]:
print(df["SeniorCitizen"].value_counts())
print("Age" in df.columns)


SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64
False


In [3]:
## Q2. The TotalCharges column is stored as an object. Identify invalid values, convert it to numeric, 
## and quantify affected records.


# Convert totalcharges to numeric 
totalcharges_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Identify invalid totalcharges
invalid_totalcharges = df[
    totalcharges_numeric.isna()
].copy()

print("\nQ2: TOTALCHARGES")
print("Invalid records:", len(invalid_totalcharges))
print(
    "Affected percentage:",
    round(len(invalid_totalcharges) / len(df) * 100, 2),
    "%"
)

print("\nInvalid values:")
print(
    df.loc[
        totalcharges_numeric.isna(),
        "TotalCharges"
    ].unique()
)

# Permanently convert
df["TotalCharges"] = totalcharges_numeric


Q2: TOTALCHARGES
Invalid records: 11
Affected percentage: 0.16 %

Invalid values:
[' ']


In [4]:
### Q3. Create a function that automatically categorizes columns into Numerical, 
### Categorical, Binary, and Identifier groups.


def categorize_columns(df, id_threshold=0.90):

    numerical = []
    categorical = []
    binary = []
    identifier = []

    for column in df.columns:

        unique_count = df[column].nunique(
            dropna=True
        )

        unique_ratio = unique_count / len(df)

        if unique_count == 2:
            binary.append(column)

        elif unique_ratio >= id_threshold:
            identifier.append(column)

        elif pd.api.types.is_numeric_dtype(
            df[column]
        ):
            numerical.append(column)

        else:
            categorical.append(column)

    return {
        "Numerical": numerical,
        "Categorical": categorical,
        "Binary": binary,
        "Identifier": identifier
    }


categories = categorize_columns(df)

print("\nQ3: COLUMN CATEGORIES")

for category, columns in categories.items():

    print(f"\n{category}:")
    print(columns)



Q3: COLUMN CATEGORIES

Numerical:
['tenure', 'MonthlyCharges']

Categorical:
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

Binary:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

Identifier:
['customerID', 'TotalCharges']


In [5]:
### Q4. Identify customers where TotalCharges differs from MonthlyCharges × Tenure by more than 10%.
###Investigate possible causes.

df["ExpectedTotalCharges"] = (
    df["MonthlyCharges"] * df["tenure"]
)

df["ChargeDifference"] = (
    df["TotalCharges"]
    - df["ExpectedTotalCharges"]
)

# Avoid division by zero
df["ChargeDifferencePct"] = np.where(
    df["ExpectedTotalCharges"] != 0,
    abs(df["ChargeDifference"])
    / df["ExpectedTotalCharges"] * 100,
    np.nan
)

# Customers with >10% difference
large_difference = df[
    df["ChargeDifferencePct"] > 10
].copy()

print("\nQ4: LARGE CHARGE DIFFERENCES")

print(
    "Customers with >10% difference:",
    len(large_difference)
)

print(
    "Percentage:",
    round(
        len(large_difference) / len(df) * 100,
        2
    ),
    "%"
)

display(
    large_difference[
        [
            "customerID",
            "tenure",
            "MonthlyCharges",
            "TotalCharges",
            "ExpectedTotalCharges",
            "ChargeDifference",
            "ChargeDifferencePct"
        ]
    ].head(20)
)


Q4: LARGE CHARGE DIFFERENCES
Customers with >10% difference: 381
Percentage: 5.41 %


,customerID,tenure,MonthlyCharges,TotalCharges,ExpectedTotalCharges,ChargeDifference,ChargeDifferencePct
21,1680-VDCWW,12,19.80,202.25,237.60,-35.35,14.877946
42,9867-JCZSP,17,20.75,418.25,352.75,65.50,18.568391
47,7760-OYPDY,2,80.65,144.15,161.30,-17.15,10.632362
69,7410-OIEDU,10,79.85,887.35,798.50,88.85,11.127113
77,5590-ZSKRV,8,54.65,482.25,437.20,45.05,10.304209
105,6180-YBIQI,5,24.30,100.20,121.50,-21.30,17.530864
124,7219-TLZHO,4,20.85,62.90,83.40,-20.50,24.580336
171,1875-QIVME,2,104.40,242.80,208.80,34.00,16.283525
223,0742-MOABM,4,50.05,179.35,200.20,-20.85,10.414585
239,9227-UAQFT,16,19.75,284.35,316.00,-31.65,10.015823


In [13]:
## Module 2: Advanced Customer Analytics

In [6]:
# Calculate churn rate by Gender, Senior Citizen, Partner, and Dependents. Rank the highest-risk customer groups.
#### CHURN RATE BY GENDER


gender_churn = (
    df.groupby("gender")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Churn_Rate")
    .sort_values("Churn_Rate", ascending=False)
)
gender_churn["Churn_Rate"] = (
    gender_churn["Churn_Rate"].map(lambda x: f"{x:.2f}%")
)



print(gender_churn )

   gender Churn_Rate
0  Female     26.92%
1    Male     26.16%


In [7]:
##### CHURN RATE BY SENIOR SITIZEN



senior_churn = (
    df.groupby("SeniorCitizen")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Churn_Rate")
    .sort_values("Churn_Rate", ascending=False)
)
senior_churn["Churn_Rate"] = (
    senior_churn ["Churn_Rate"].map(lambda x : f"{x:.2f}%")
)

print(senior_churn)

   SeniorCitizen Churn_Rate
1              1     41.68%
0              0     23.61%


In [8]:
###### CHURN RATE BY PARTNER
partner_churn = (
    df.groupby("Partner")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Churn_Rate")
    .sort_values("Churn_Rate", ascending=False)
)

print(partner_churn)

  Partner  Churn_Rate
0      No   32.957979
1     Yes   19.664903


In [9]:
dependents_churn = (
    df.groupby("Dependents")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Churn_Rate")
    .sort_values("Churn_Rate", ascending=False)
)

print(dependents_churn)

  Dependents  Churn_Rate
0         No   31.279140
1        Yes   15.450237


In [10]:
#### Combine All Four for Risk Assessment 

def churn_rate_by_column(df, column):
    
    result = (
        df.groupby(column)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .reset_index(name="Churn_Rate")
        .sort_values("Churn_Rate", ascending=False)
    )
    
    return result


for column in [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents"
]:
    
    print("\n", "=" * 50)
    print(column)
    print("=" * 50)
    
    print(churn_rate_by_column(df, column))


gender
   gender  Churn_Rate
0  Female   26.920872
1    Male   26.160338

SeniorCitizen
   SeniorCitizen  Churn_Rate
1              1   41.681261
0              0   23.606168

Partner
  Partner  Churn_Rate
0      No   32.957979
1     Yes   19.664903

Dependents
  Dependents  Churn_Rate
0         No   31.279140
1        Yes   15.450237


In [14]:
#### Rank Them


results = []

for column in [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents"
]:
    
    temp = churn_rate_by_column(df, column)
    temp["Variable"] = column
    temp["Group"] = temp[column].astype(str)
    
    results.append(
        temp[["Variable", "Group", "Churn_Rate"]]
    )

risk_groups = pd.concat(results)

risk_groups = risk_groups.sort_values(
    "Churn_Rate",
    ascending=False
)

print(risk_groups)

        Variable   Group  Churn_Rate
1  SeniorCitizen       1   41.681261
0        Partner      No   32.957979
0     Dependents      No   31.279140
0         gender  Female   26.920872
1         gender    Male   26.160338
0  SeniorCitizen       0   23.606168
1        Partner     Yes   19.664903
1     Dependents     Yes   15.450237


In [19]:
df["TenureSegment"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 60, float("inf")],
    labels=[
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49-60 months",
        "61+ months"
    ]
)

print(df["TenureSegment"].value_counts().sort_index())

TenureSegment
0-12 months     2186
13-24 months    1024
25-48 months    1594
49-60 months     832
61+ months      1407
Name: count, dtype: int64


In [21]:
tenure_churn = (
    df.groupby("TenureSegment", observed=False)["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Churn_Rate")
)

print(tenure_churn)

  TenureSegment  Churn_Rate
0   0-12 months   47.438243
1  13-24 months   28.710938
2  25-48 months   20.388959
3  49-60 months   14.423077
4    61+ months    6.609808


In [22]:
## Rank the segments

tenure_churn = tenure_churn.sort_values(
    "Churn_Rate",
    ascending=False
)

print(tenure_churn)

  TenureSegment  Churn_Rate
0   0-12 months   47.438243
1  13-24 months   28.710938
2  25-48 months   20.388959
3  49-60 months   14.423077
4    61+ months    6.609808


In [39]:
## Customer Count Across Segment

tenure_analysis = (
    df.groupby("TenureSegment", observed=False)
    .agg(
        Customers=("customerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

tenure_analysis["Churn_Rate"] = (
    tenure_analysis["Churned"]
    / tenure_analysis["Customers"]
    * 100
)

print(tenure_analysis)

  TenureSegment  Customers  Churned  Churn_Rate
0   0-12 months       2186     1037   47.438243
1  13-24 months       1024      294   28.710938
2  25-48 months       1594      325   20.388959
3  49-60 months        832      120   14.423077
4    61+ months       1407       93    6.609808


In [43]:
###  Q8. Identify the top 10 customer profiles most likely to churn using combinations of Contract, 
### Payment Method, and Internet Service.

## Group by Combinations 
profile_analysis = (
    df.groupby(
        [
            "Contract",
            "PaymentMethod",
            "InternetService"
        ]
    )
    .agg(
        Customers=("customerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)
### Calculate Churn rate 

profile_analysis["Churn_Rate"] = (
    profile_analysis["Churned"]
    / profile_analysis["Customers"]
    * 100
)

top_10_profiles = profile_analysis.sort_values(
    "Churn_Rate",
    ascending=False
).head(10)

print(top_10_profiles)

          Contract              PaymentMethod InternetService  Customers  \
7   Month-to-month           Electronic check     Fiber optic       1307   
10  Month-to-month               Mailed check     Fiber optic        201   
1   Month-to-month  Bank transfer (automatic)     Fiber optic        327   
4   Month-to-month    Credit card (automatic)     Fiber optic        293   
6   Month-to-month           Electronic check             DSL        474   
9   Month-to-month               Mailed check             DSL        367   
3   Month-to-month    Credit card (automatic)             DSL        185   
19        One year           Electronic check     Fiber optic        196   
11  Month-to-month               Mailed check              No        325   
2   Month-to-month  Bank transfer (automatic)              No         65   

    Churned  Churn_Rate  
7       789   60.367253  
10      102   50.746269  
1       149   45.565749  
4       122   41.638225  
6       192   40.506329  
9      